# Check Loading the VOC2012 Segmentation Dataset for Downstream Tasks

In [11]:
from PIL import Image
from torchvision import transforms
from torch.utils.data import DataLoader

from oxels.datasets.voc2012_dataset import VOCDataset
# Define input image transformations
input_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Define target (segmentation mask) transformations
# Using NEAREST interpolation to avoid smoothing label values
target_transform = transforms.Compose([
    transforms.Resize((256, 256), interpolation=Image.NEAREST),
    transforms.PILToTensor(),  # Produces 1 x H x W tensor with class indices
])

# Create the VOC2012 segmentation datasets

train_dataset = VOCDataset(
    year='2012',
    image_set='train',
    download=False,
    transform=input_transform,
    target_transform=target_transform
)

val_dataset = VOCDataset(
    year='2012',
    image_set='val',
    download=True,
    transform=input_transform,
    target_transform=target_transform
)

# Wrap datasets in DataLoaders
batch_size = 8
num_workers = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    #num_workers=num_workers,
    #pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    #num_workers=num_workers,
    #pin_memory=False
)

# Check the dataset sizes
print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of validation samples: {len(val_dataset)}")

Number of training samples: 1464
Number of validation samples: 1449


# VOC2012 Model

In [12]:
import os
from oxels.models import ImageNetModel


backbone_run_name = "amber-terrain-77"
#backbone_run_name = "scarlet-sky-138"
backbone_run_name = "last.ckpt"
#ckpt_dir = os.path.join("../checkpoints")
backbone_ckpt_path = os.path.join("../checkpoints", backbone_run_name)
backbone = ImageNetModel.load_from_checkpoint(backbone_ckpt_path)
print(f"Loaded model from {os.path.join("../checkpoints", backbone_run_name)}")
num_oxels = backbone.hparams.num_oxels

Loaded model from ../checkpoints/last.ckpt


# Some Pixel Classifier

In [13]:
import torch
import torch.nn as nn
from torchmetrics.classification import Accuracy
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from oxels.models.metrics_mixin import MetricsMixin
import torch.nn.functional as F
import lightning as L


class Simple1x1Classifier(nn.Module):
    def __init__(self, in_channels: int, num_classes: int = 1, head_groups: int = 1):
        super().__init__()
        self.num_classes = num_classes
        self.linear = nn.Linear(in_channels, num_classes)

    def forward(self, x):
        x = x.permute(0, 2, 3, 1)
        x = self.linear(x)
        x = x.permute(0, 3, 1, 2)  # [B, C, H, W]
        return x

class MLPStefanClassifier(nn.Module):
    def __init__(self, in_channels: int, num_classes: int = 1):
        super().__init__()
        self.num_classes = num_classes
        self.mlp = nn.Sequential(
            nn.Linear(in_channels, 150),
            nn.ReLU(),
            nn.Linear(150, 100),
            nn.ReLU(),
            nn.Linear(100, 50),
            nn.ReLU(),
            nn.Linear(50, num_classes),
        )

    def forward(self, x):
        x = x.permute(0, 2, 3, 1)  # [B, H, W, C]
        x = self.mlp(x)
        x = x.permute(0, 3, 1, 2)  # [B, C, H, W]
        return x


class DGLinearClassifier(MetricsMixin, L.LightningModule):
    def __init__(
        self,
        backbone: nn.Module,
        head: nn.Module,
        dataset_name: str,
        train_batch_size: int,
        val_batch_size: int,
        num_oxels: int = 64,
        image_size: int = 256,
        weight_decay: float = 0.004,
        learning_rate: float = 1e-3,
        lr_pct_start: float = 0.05,
        lr_div_factor: float = 25.0,
        lr_final_div_factor: float = 1e4,
    ):
        super().__init__()
        self.save_hyperparameters(
            ignore=["backbone", "head", "dataset_name"],
        )
        self.backbone = backbone
        self.backbone.freeze()
        self.head = head
        self.dataset_name = dataset_name
        num_classes = head.num_classes
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=255)
        self.train_acc = Accuracy(task="multiclass", num_classes=num_classes, ignore_index=255)
        self.val_acc = Accuracy(task="multiclass", num_classes=num_classes, ignore_index=255)
        self.test_acc = Accuracy(task="multiclass", num_classes=num_classes, ignore_index=255)

    def forward(self, x):
        oxels = self.backbone(x)
        prediction = self.head(oxels)
        return prediction

    def compute_loss(self, batch):
        images, labels = batch # Bx3xHxW,  Bx1xHxW
        logits = self(images) # bx21xhxw
        labels = labels.to(dtype=logits.dtype).squeeze(1) # BxHxW
        loss = self.loss_fn(logits, labels)
        return loss

    def compute_metrics(self, batch):
        images, labels = batch
        logits = self(images)
        B, C, H, W = logits.shape

        # Resize labels to match logits spatial dimensions
        #labels_resized = F.interpolate(labels.float(), size=(H, W), mode="nearest").squeeze(1).long()
        labels = labels.squeeze(1).long()
        loss = self.loss_fn(logits, labels)

        preds = torch.argmax(logits, dim=1) # BxHxW

        # choose the right Accuracy object based on stage
        if self.trainer.training:
            acc = self.train_acc(preds, labels)
        elif self.trainer.validating:
            acc = self.val_acc(preds, labels)
        else:  # testing
            acc = self.test_acc(preds, labels)

        return {
            "loss": loss,
            "accuracy": acc,
        }

    def configure_optimizers(self):
        lr = self.hparams.learning_rate
        wd = self.hparams.weight_decay

        # only use parameters that requires grad
        params = [p for p in self.parameters() if p.requires_grad]
        optimizer = AdamW(params, lr=lr, weight_decay=wd, betas=(0.9, 0.99))
        scheduler = OneCycleLR(
            optimizer,
            max_lr=lr,
            total_steps=self.trainer.estimated_stepping_batches,
            div_factor=self.hparams.lr_div_factor,
            final_div_factor=self.hparams.lr_final_div_factor,
            pct_start=self.hparams.lr_pct_start,
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "interval": "step"},
        }

    def validation_step(self, batch, batch_idx, dataloader_idx=0):
        metrics = self.compute_metrics(batch)
        prefix = "validation/" if dataloader_idx == 0 else "validation/ood/"
        for key, value in metrics.items():
            key = f"{prefix}{key}"
            self.log(key,
                     value,
                     on_step=False,
                     on_epoch=True,
                     sync_dist=True,
                     prog_bar=(dataloader_idx == 0),  # maybe only show ID in prog bar
                     logger=True)

        return metrics["loss"]

    def train_dataloader(self):
        # Create the VOC2012 segmentation datasets
        train_dataset = VOCDataset(
            year='2012',
            image_set='train',
            download=False,
            transform=input_transform,
            target_transform=target_transform
        )
        return DataLoader(
            train_dataset,
            batch_size=self.hparams.train_batch_size,
            shuffle=True,
            pin_memory=True,
            num_workers=len(os.sched_getaffinity(0)),
            drop_last=True,
        )

    def val_dataloader(self):

        val_dataset = VOCDataset(
            year='2012',
            image_set='val',
            download=True,
            transform=input_transform,
            target_transform=target_transform
        )


        return DataLoader(val_dataset, batch_size=self.hparams.val_batch_size, shuffle=False,  num_workers=len(os.sched_getaffinity(0)), drop_last=False)

In [14]:
import torch
from lightning.pytorch import callbacks
from lightning.pytorch.loggers import WandbLogger
from torchmetrics import Accuracy
import lightning as L
from lightning.pytorch.loggers import WandbLogger

import wandb
from oxels.callbacks import ShowOxels


torch.cuda.empty_cache()
torch.set_float32_matmul_precision("medium")

print("Device Count:", torch.cuda.device_count())
project = "VOCS_synthetic-testing"
total_steps = 3_000
image_size = 256
num_nodes = 1
num_devices = 1
train_batch_size = 64
val_batch_size = 128
learning_rate = 2e-3
lr_pct_start = 0.05
weight_decay = 1e-6

model_config = dict(
    num_oxels=num_oxels,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    lr_pct_start=lr_pct_start,
    lr_div_factor=25.0,
    lr_final_div_factor=1e3,
    train_batch_size=train_batch_size,
    val_batch_size=val_batch_size,
    image_size=image_size,
)

trainer_config = dict(
    gradient_clip_val=3.0,
    gradient_clip_algorithm="value",
    max_steps=total_steps,
    accelerator="gpu",
    strategy="auto",
    devices=num_devices,
    precision="16-mixed",
    num_nodes=num_nodes,
)

model = DGLinearClassifier(
    backbone=backbone,
    head=MLPStefanClassifier(num_oxels, 21),
    dataset_name="synthetic_VOCS",
    train_batch_size=train_batch_size,
    val_batch_size=val_batch_size,
    num_oxels=num_oxels,
    image_size=image_size,
    weight_decay=weight_decay,
    learning_rate=learning_rate,
    lr_pct_start=lr_pct_start,
    lr_div_factor=25.0,
    lr_final_div_factor=1e4)

with torch.device("cpu"):
    train_images = [model.train_dataloader().dataset[i][0] for i in range(4)]
    train_images = torch.stack(train_images)
    validation_images = [model.val_dataloader().dataset[i][0] for i in range(4)]
    validation_images = torch.stack(validation_images)

num_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total parameters: ", num_parameters)
config = model_config | trainer_config

Device Count: 1
Total parameters:  30971


In [ ]:
run = wandb.init(
    entity="kl_divergence-rensselaer-polytechnic-institute",
    project=project,
    config=config,
    dir="wandb_results"
)

logger = WandbLogger(
    experiment=run,
)

wandb.summary["num_parameters"] = num_parameters
wandb.summary["backbone_run_id"] = "synthetic_VOCS_1.0"
wandb.summary["backbone_run_name"] = backbone_run_name
wandb.summary["model_type"] = "DGLinearClassifier"
run_name = wandb.run.name
run_id = wandb.run.id
print(f"name: {run_name} \t run id:{run_id}")

ckpt_dir = os.path.join("lightning_logs", str(run_name))
# make sure the directory exists
os.makedirs(ckpt_dir, exist_ok=True)
with open(os.path.join(ckpt_dir, "wandb_run_id.txt"), "w") as f:
    f.write(run_id)
trainer = L.Trainer(
    **trainer_config,
    callbacks=[
        callbacks.LearningRateMonitor(logging_interval="step"),
        callbacks.ModelCheckpoint(
            dirpath=ckpt_dir,
            monitor="validation/loss", 
            mode="min",
            save_top_k=1,
            filename="best_model",
            save_last=True,
        ),
        # ShowOxels(images=train_images, every_n_epochs=10, caption="Train Oxels"),
        # ShowOxels(images=validation_images, every_n_epochs=10, caption="Validation Oxels"),
        # ShowOxels(images=validation_images_ood, every_n_epochs=10, caption="Validation OOD Oxels"),
    ],
    logger=logger,
)

try:
    trainer.fit(model)
    result = trainer.callback_metrics["validation/loss"]
    wandb.summary["result"] = result

finally:
    # clean up
    wandb.finish()

wandb: Currently logged in as: averkleeren (averkleeren-rensselaer-polytechnic-institute) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/verkla/Desktop/Research/oxels/.venv/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /home/verkla/Desktop/Research/oxels/notebooks/lightning_logs/vocal-water-1 exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type                | Params | Mode 
----------------------------------------------------------
0 | backbone  | ImageNetModel       | 977 K  | eval 
1 | head      | MLPStefanClassifier | 31.0 K | train
2 | loss_fn   | CrossEntropyLoss    | 0      | train
3 | train_acc | MulticlassAccuracy  | 0      | train
4 | val_acc   | MulticlassAccuracy  | 0      | train
5 | test_acc  | MulticlassAccuracy  | 0      | train
----------------------------------------------------------
31.0 K    Trainable params
977 K     Non-trainable params
1.0 M

name: vocal-water-1 	 run id:mv2puzby


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/verkla/Desktop/Research/oxels/.venv/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:310: The number of training batches (22) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]